In [1]:
# imports
import torch
import re
import matplotlib.pyplot as plt
from pathlib import Path
import tqdm

import pandas as pd
import os
os.environ["OMP_NUM_THREADS"] = "4"  # export OMP_NUM_THREADS=4
os.environ["OPENBLAS_NUM_THREADS"] = "4"  # export OPENBLAS_NUM_THREADS=4
os.environ["MKL_NUM_THREADS"] = "6"  # export MKL_NUM_THREADS=6
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"  # export VECLIB_MAXIMUM_THREADS=4
os.environ["NUMEXPR_NUM_THREADS"] = "6"  # export NUMEXPR_NUM_THREADS=6

from checkpoints_to_datasets.dataset_base_optimized import ModelDatasetBase

/Users/ricostadeli/Documents/HSG/thesis/ms-thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-09 19:51:46,593	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
def load_dataset_from_path(
    path: Path,
    dstype: str,
    epoch_lst=None,
    preload_properties=False,
    validate_checkpoints=False,
):
    """
    Loads lazy dataset objects from raw zoo.

    Args:
        path: pathlib.Path to raw model zoo
        dstype: "Seed", "Hyp-fix", or "Hyp-rand"
        dssize: "small" or "large"
        epoch_lst: list of epochs to index
        preload_properties: whether to read result.json metadata at init
        validate_checkpoints: whether to fully validate all checkpoints during init

    Returns:
        dict with "trainset", "valset", "testset"
    """
    if epoch_lst is None:
        epoch_lst = [5, 15, 25]

    result_key_list = [
        "test_acc",
        "training_iteration",
        "ggap",
    ]

    if dstype == "Seed":
        config_key_list = []
    elif dstype == "ViT":
        config_key_list = [
            "optim::optimizer",
            "model::dropout",
            "optim::lr",
            "optim::wd",
        ]
    else:
        config_key_list = [
            "model::nlin",
            "model::init_type",
            "optim::optimizer",
            "model::dropout",
            "optim::lr",
            "optim::wd",
        ]

    property_keys = {
        "result_keys": result_key_list,
        "config_keys": config_key_list,
    }

    path_zoo_root = [path.absolute()]

    common_kwargs = dict(
        root=path_zoo_root,
        layer_lst=None,
        epoch_lst=epoch_lst,
        use_bias=True,
        ds_split=None, # To use a split do [0.7,0.3]
        max_samples=None,
        weight_threshold=5,
        filter_function=None,
        property_keys=property_keys,
        num_threads=6,
        verbosity=0,
        shuffle_path=True,
        preload_properties=preload_properties,
        validate_checkpoints=validate_checkpoints,
    )

    # trainset = ModelDatasetBase(train_val_test="train", **common_kwargs)
    # valset = ModelDatasetBase(train_val_test="val", **common_kwargs)
    # testset = ModelDatasetBase(train_val_test="test", **common_kwargs)
    full = ModelDatasetBase(train_val_test="full", **common_kwargs)

    return {
        "trainset": full,
    }

In [3]:
dspath_raw = Path("../../data/ViT")
dstype = "ViT"
# dspath_raw = Path("../../data/tune_zoo_cifar10_small_hyperparameter_10_random_seeds")
# dstype = "Hyp-rand"
epoch_lst = list(range(0, 51))

ds_custom = load_dataset_from_path(
    path=dspath_raw,
    dstype=dstype,
    epoch_lst=epoch_lst,
    preload_properties=True,
    validate_checkpoints=False,
)

trainset = ds_custom["trainset"]
print(len(trainset))

Loading properties: 100%|██████████| 11/11 [00:00<00:00, 452.49it/s]

11


In [4]:
def infer_layers_from_checkpoint(checkpoint):
    """
    Infer available layers from checkpoint keys.

    Returns
    -------
    list[tuple[int, str]]
        List of (layer_idx, layer_type), sorted by layer_idx.
    """
    layer_indices = set()
    pattern = re.compile(r"^module_list\.(\d+)\.weight$")

    for key in checkpoint.keys():
        match = pattern.match(key)
        if match:
            layer_indices.add(int(match.group(1)))

    inferred = []
    for layer_idx in sorted(layer_indices):
        w_key = f"module_list.{layer_idx}.weight"
        w = checkpoint[w_key]

        if w.ndim == 4:
            layer_type = "conv2d"
        elif w.ndim == 2:
            layer_type = "fc"
        else:
            layer_type = f"{w.ndim}d"

        inferred.append((layer_idx, layer_type))

    return inferred


def compute_per_layer_norms(dataset, combine_weight_and_bias=True, show_progress=True):
    """
    Compute per-layer L1, L2, and Linf norms for every checkpoint in a lazy dataset.

    Works even if dataset.layer_lst is None, by inferring layers from checkpoint keys.
    """
    rows = []

    iterator = range(len(dataset))
    if show_progress:
        iterator = tqdm.tqdm(iterator, total=len(dataset), desc="Computing norms")

    for i in iterator:
        sample = dataset.samples[i]
        path = sample["path"]

        # supports both old and simplified dataset formats
        if "epoch" in sample:
            epoch = sample["epoch"]
        else:
            epoch = sample["epoch_in"]

        checkpoint = dataset._load_checkpoint(path, epoch)

        row = {
            "sample_idx": i,
            "path": str(path),
            "epoch": int(epoch),
        }

        if dataset.properties is not None:
            for key, values in dataset.properties.items():
                row[key] = values[i]

        if getattr(dataset, "layer_lst", None) is not None:
            layers = dataset.layer_lst
        else:
            layers = infer_layers_from_checkpoint(checkpoint)

        for layer_idx, layer_type in layers:
            w_key = f"module_list.{layer_idx}.weight"
            b_key = f"module_list.{layer_idx}.bias"

            if w_key not in checkpoint:
                continue

            w = checkpoint[w_key].detach().float().reshape(-1)
            prefix = f"layer_{layer_idx}_{layer_type}"

            row[f"{prefix}_weight_only_l1"] = torch.linalg.vector_norm(w, ord=1).item()
            row[f"{prefix}_weight_only_l2"] = torch.linalg.vector_norm(w, ord=2).item()
            row[f"{prefix}_weight_only_linf"] = torch.linalg.vector_norm(
                w, ord=float("inf")
            ).item()

            if combine_weight_and_bias and getattr(dataset, "use_bias", True) and b_key in checkpoint:
                b = checkpoint[b_key].detach().float().reshape(-1)
                x = torch.cat([w, b], dim=0)
            else:
                x = w

            row[f"{prefix}_l1"] = torch.linalg.vector_norm(x, ord=1).item()
            row[f"{prefix}_l2"] = torch.linalg.vector_norm(x, ord=2).item()
            row[f"{prefix}_linf"] = torch.linalg.vector_norm(x, ord=float("inf")).item()

        rows.append(row)
        del checkpoint

    return pd.DataFrame(rows)

In [5]:
trainset = ds_custom["trainset"]

df_norms = compute_per_layer_norms(trainset, combine_weight_and_bias=True)
df_norms.head()

Computing norms: 100%|██████████| 11/11 [00:00<00:00, 175.85it/s]


,sample_idx,path,epoch,test_acc,training_iteration,ggap,optim::optimizer,model::dropout,optim::lr,optim::wd
0,0,/Users/ricostadeli/Documents/HSG/thesis/ms-the...,0,0.1000,0,0.00000,adamw,0.1,0.006,0.0005
1,1,/Users/ricostadeli/Documents/HSG/thesis/ms-the...,5,0.2715,5,-0.02412,adamw,0.1,0.006,0.0005
2,2,/Users/ricostadeli/Documents/HSG/thesis/ms-the...,10,0.3039,10,-0.01060,adamw,0.1,0.006,0.0005
3,3,/Users/ricostadeli/Documents/HSG/thesis/ms-the...,15,0.3475,15,-0.01108,adamw,0.1,0.006,0.0005
4,4,/Users/ricostadeli/Documents/HSG/thesis/ms-the...,20,0.3687,20,0.00454,adamw,0.1,0.006,0.0005


In [6]:
example_path = df_norms["path"].iloc[0]
df_one_model = df_norms[df_norms["path"] == example_path].sort_values("epoch")


In [7]:
df_plot = df_one_model[df_one_model["epoch"] != 0]

plt.plot(df_plot["epoch"], df_plot["layer_0_conv2d_l2"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("L2 norm")
plt.title("Layer 0 conv2d L2 trajectory")
plt.show()

KeyError: 'layer_0_conv2d_l2'

# Load ViT Data

In [ ]:
chkpth = Path("../../data/ViT/NN_tune_trainable_541f4_00000_0_model_embedding_dim=6,optim_lr=0.0060,optim_wd=0.0005,seed=0,training_batchsize=512_2024-08-19_07-33-29/checkpoint_000000/checkpoints")
ckpt = torch.load(str(chkpth), map_location=torch.device("cpu"))

In [ ]:
state_dict = ckpt["model_state_dict"]